##### Task 12
Build an item performance table.

What I want in the end:
- One row per item
- Columns including:
  - `item_id`
  - `product_name`
  - total quantity sold
  - total revenue
  - number of distinct orders containing the item

In [0]:
%sql
USE SCHEMA e_commerce;

In [0]:
%sql
-- First step from orders tables: order_id, item_id, product_name, quantity, price, revenue
-- Second step from first view: item_id, product_name, total_quantity, total_revenue, distict_orders

CREATE OR REPLACE TEMP VIEW order_details AS
SELECT o.order_id, o.item_id, i.product_name, i.price, o.quantity, i.price * o.quantity revenue
FROM (
  SELECT order_id, item.item_id, item.quantity
  FROM silver_orders
  LATERAL VIEW explode(items) AS item
) o
LEFT JOIN silver_items i
  ON o.item_id = i.item_id;

SELECT *
FROM order_details;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW item_performance AS
SELECT 
  i.item_id, 
  i.product_name, 
  SUM(o.quantity) total_quantity_sold,
  SUM(o.revenue) total_revenue,
  COUNT(DISTINCT(o.order_id)) distinct_orders
FROM silver_items i
LEFT JOIN order_details o
  ON i.item_id = o.item_id
GROUP BY i.item_id, i.product_name
HAVING distinct_orders > 0;

SELECT *
FROM item_performance;